
# LoRA Fine-Tuning Qwen2.5 1.5B-Instruct V3
This notebook fine-tunes a HuggingFace Qwen2.5 1.5B-Instruct model using LoRA (PEFT).

In [4]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata
userdata.get('login_key')

Mounted at /content/drive


'nona-yo_bid-niss'

In [5]:
!pip install -q transformers peft evaluate tomli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 12.3 MB/s eta 0:00:00


In [6]:
!pip install evaluate
import evaluate
print("Evaluate library version:", evaluate.__version__)

Evaluate library version: 0.4.6


In [7]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [8]:
import os
import re
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score
import evaluate

from peft import LoraConfig, get_peft_model

In [9]:
torch.set_default_dtype(torch.float32)


In [10]:
import evaluate
import datetime
import re

# Initialize the metric
f1_metric = evaluate.load("f1")

def format_time(elapsed):
    elapsed_rounded = int(round(elapsed))
    return str(datetime.timedelta(seconds=elapsed_rounded))

def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    return text.replace("\n", " ")

def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(
        f"trainable params: {trainable_params} || "
        f"all params: {all_param} || "
        f"trainable%: {100 * trainable_params / all_param:.2f}"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [15]:
class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=True,
        use_lora=False,
        is_base_encoder=True,
        lora_rank=16,
        lora_alpha=16,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)
        self.is_base_encoder = is_base_encoder

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        if use_lora:
            lora_config = LoraConfig(
                r=lora_rank,
                lora_alpha=lora_alpha,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                lora_dropout=0.2,
                bias="none",
                task_type="FEATURE_EXTRACTION"
            )
            self.model = get_peft_model(self.model, lora_config)

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        if self.is_base_encoder:
            pooled_output = outputs.pooler_output
        else:
            # Use mean pooling or CLS token if pooler is not available
            pooled_output = outputs.last_hidden_state[:, -1, :]

        pooled_output = pooled_output.to(torch.device("cuda"), dtype=torch.float)
        logits = self.classifier(pooled_output)
        return logits

In [13]:
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.amp import autocast, GradScaler

class TrainerModule:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        epochs,
        lr,
        patience,
        output_dir,
        use_bf16=False,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        self.use_bf16 = use_bf16
        if self.use_bf16:
            self.scaler = GradScaler('cuda')

    def train(self, start_epoch=0):
        for epoch in range(start_epoch, self.epochs):
            print(f"\nEpoch {epoch+1}/{self.epochs}")
            self.model.train()

            total_loss, total_acc = 0, 0
            # Changed print_every to 10 for more frequent feedback
            print_every = 10

            for batch_idx, batch in enumerate(tqdm(self.train_loader, desc=f"Training Epoch {epoch+1}")):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                with autocast(device_type='cuda', dtype=torch.bfloat16, enabled=self.use_bf16):
                    logits = self.model(input_ids, attention_mask)
                    loss = self.loss_fn(logits.squeeze(1), labels)

                if self.use_bf16:
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    loss.backward()
                    self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy().flatten()
                total_acc += accuracy_score(labels.cpu().numpy(), (preds >= 0.5).astype(int))

                if (batch_idx + 1) % print_every == 0:
                    print(f"Batch {batch_idx + 1}/{len(self.train_loader)} | "
                          f"Running Loss: {total_loss / (batch_idx + 1):.4f} | "
                          f"Running Acc: {total_acc / (batch_idx + 1):.4f}")

            avg_train_loss = total_loss / len(self.train_loader)
            avg_train_acc = total_acc / len(self.train_loader)
            self.history['train_loss'].append(avg_train_loss)
            self.history['train_acc'].append(avg_train_acc)

            print(f"End Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f}")
            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()
        total_loss, total_acc = 0, 0

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Evaluating"):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device)

                with autocast(device_type='cuda', dtype=torch.bfloat16, enabled=self.use_bf16):
                    logits = self.model(input_ids, attention_mask)
                    loss = self.loss_fn(logits.squeeze(1), labels)

                total_loss += loss.item()
                preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy().flatten()
                total_acc += accuracy_score(labels.cpu().numpy(), (preds >= 0.5).astype(int))

        avg_val_loss = total_loss / len(self.val_loader)
        avg_val_acc = total_acc / len(self.val_loader)
        self.history['val_loss'].append(avg_val_loss)
        self.history['val_acc'].append(avg_val_acc)

        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}")

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'history': self.history
        }
        torch.save(checkpoint, os.path.join(self.output_dir, f"checkpoint_epoch_{epoch}.pt"))
        tokenizer.save_pretrained(self.output_dir)
        print(f"Checkpoint saved to {self.output_dir}")

In [11]:
class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        use_decomp,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.use_decomp = use_decomp

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Add the same tokens used during training to match embedding size
        CRITICAL_NEGATIONS = [
            'مش', 'مو', 'مانو', 'مب', 'ليس', 'ما', 'لا', 'لم', 'لن',
            'ماني', 'مانك', 'مانو', 'مانها', 'موصلش', 'مبيش'
        ]
        self.tokenizer.add_tokens(CRITICAL_NEGATIONS)

        self.model = CustomClassifier(base_model, use_lora=True, is_base_encoder=False)

        # Resize embeddings to match the updated tokenizer (151936)
        self.model.model.resize_token_embeddings(len(self.tokenizer))

        # Load the full checkpoint dictionary
        checkpoint = torch.load(model_path, map_location=self.device)

        # Extract only the model's state_dict
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        self.model.eval()

    def encode_input(self, claim, questions, verdict, justification, max_length=150):

        text = f"Claim: {claim}\nVerdict: {verdict}\nJustification: {justification}"

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        return (
            encoding["input_ids"].to(self.device),
            encoding["attention_mask"].to(self.device),
        )

    def score(self, claim, questions, verdict, justification):
        ids, mask = self.encode_input(claim, questions, verdict, justification)
        with torch.no_grad():
            return float(self.model(ids, mask).item())

In [1]:
# ===== PATH PLACEHOLDERS =====
TRAIN_CSV = "/content/drive/MyDrive/output/Arabic_train_preprocessed.jsonl"
VAL_CSV   = "/content/drive/MyDrive/output/Arabic_val_preprocessed.jsonl"
TEST_JSON = "/content/drive/MyDrive/output/Arabic_test_preprocessed.json"

OUTPUT_DIR = "/content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models"

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_LENGTH = 800
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-4

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Ensure you have run the cells defining TextDataset, CustomClassifier, and TrainerModule first
login(userdata.get('login_key'))

train_df = pd.read_json(TRAIN_CSV, lines = True)
val_df = pd.read_json(VAL_CSV, lines = True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
CRITICAL_NEGATIONS = [
    'مش', 'مو', 'مانو', 'مب', 'ليس', 'ما', 'لا', 'لم', 'لن',
    'ماني', 'مانك', 'مانو', 'مانها', 'موصلش', 'مبيش'
]
# 3. Add them to the tokenizer
num_added_toks = tokenizer.add_tokens(CRITICAL_NEGATIONS)
print(f"Added {num_added_toks} tokens")

# 4. VERY IMPORTANT: Resize the model's embeddings to match the new tokenizer


train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

lora_rank_value = 16
lora_alpha_value = lora_rank_value

model = CustomClassifier(
    BASE_MODEL,
    use_lora=True,
    is_base_encoder=False,
    lora_rank=lora_rank_value,
    lora_alpha=lora_alpha_value,
)
# Access the internal HuggingFace model to resize embeddings
model.model.resize_token_embeddings(len(tokenizer))

print_trainable_parameters(model)

trainer = TrainerModule(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LR,
    patience=2,
    output_dir=OUTPUT_DIR,
    use_bf16=True,
)

# Note: You can comment out trainer.train() if you only want to initialize and then load a checkpoint
trainer.train()

Added 14 tokens


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18466305 || all params: 1561785857 || trainable%: 1.18


/tmp/ipykernel_4325/4039747111.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()



Epoch 1/3


...

Batch 300/316 | Running Loss: 0.0022 | Running Acc: 0.9993


/tmp/ipykernel_4325/4039747111.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16, enabled=self.use_bf16):
Training Epoch 3:  95%|█████████▌| 301/316 [07:34<00:22,  1.51s/it]/tmp/ipykernel_4325/4039747111.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16, enabled=self.use_bf16):
Training Epoch 3:  96%|█████████▌| 302/316 [07:36<00:21,  1.51s/it]/tmp/ipykernel_4325/4039747111.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16, enabled=self.use_bf16):
Training Epoch 3:  96%|█████████▌| 303/316 [07:37<00:19,  1.51s/it]/tmp/ipykernel_4325/4039747111.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autoc

End Epoch 3 | Train Loss: 0.0021 | Train Acc: 0.9993


Evaluating:   0%|          | 0/78 [00:00<?, ?it/s]/tmp/ipykernel_4325/4039747111.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16, enabled=self.use_bf16):
Evaluating:   1%|▏         | 1/78 [00:00<00:43,  1.78it/s]/tmp/ipykernel_4325/4039747111.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16, enabled=self.use_bf16):
Evaluating:   3%|▎         | 2/78 [00:01<00:42,  1.77it/s]/tmp/ipykernel_4325/4039747111.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16, enabled=self.use_bf16):
Evaluating:   4%|▍         | 3/78 [00:01<00:42,  1.77it/s]/tmp/ipykernel_4325/4039747111.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Plea

Val Loss: 0.2023 | Val Acc: 0.9567
Checkpoint saved to /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models


### Normalize Labels to Numerical Values
This step ensures that the 'Class' labels in the training and validation data, as well as any existing 'label' fields in the test data, are converted to consistent numerical values. Specifically, 'True' is mapped to 1, 'False' to 0, and any string representations of numbers (including Eastern Arabic numerals) are converted to integers. Note that the current model is configured for binary classification, meaning it expects labels to be 0 or 1. If your dataset contains more than two classes, the model architecture (e.g., `num_labels` in `CustomClassifier` and the loss function) would need to be adjusted.

In [ ]:
def convert_eastern_to_western_arabic(text):
    if not isinstance(text, str):
        return text
    eastern_to_western = {
       # Standard Eastern Arabic (Mashriq)
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4',
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9',
        # Persian/Urdu variants (Extended Arabic-Indic)
        '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
        '۵': '5', '۶': '6', '٧': '7', '٨': '8', '٩': '9',
        # Regional Punctuation
        '٫': '.',
        '٬': '' # Usually better to strip thousands separators for math
    }
    return ''.join([eastern_to_western.get(char, char) for char in text])

def normalize_numerical_label(label_value):
    if isinstance(label_value, str):
        label_value = convert_eastern_to_western_arabic(label_value)
        label_value_lower = label_value.lower()
        if label_value_lower == "true":
            return 1
        elif label_value_lower == "false":
            return 0
        try:
            return int(label_value_lower)
        except ValueError:
            pass
    return label_value

train_df = pd.read_json(TRAIN_CSV, lines = True)
val_df = pd.read_json(VAL_CSV, lines = True)

# 1. Normalize Labels
train_df['Class'] = train_df['Class'].apply(normalize_numerical_label)
val_df['Class'] = val_df['Class'].apply(normalize_numerical_label)

# 2. Convert 'input_text' numerals to Western Arabic
train_df['input_text'] = train_df['input_text'].apply(convert_eastern_to_western_arabic)
val_df['input_text'] = val_df['input_text'].apply(convert_eastern_to_western_arabic)

# 3. Process test_data recursively to convert all strings containing numerals
def deep_convert_dict(obj):
    if isinstance(obj, dict):
        return {k: deep_convert_dict(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [deep_convert_dict(i) for i in obj]
    elif isinstance(obj, str):
        return convert_eastern_to_western_arabic(obj)
    else:
        return obj

with open(TEST_JSON, "r") as f:
    test_data = json.load(f)
test_data = deep_convert_dict(test_data)

print("Preprocessing complete (Labels + Text conversion).")

Preprocessing complete (Labels + Text conversion).


### Saving Normalized Data to Files
Now that the labels in `train_df`, `val_df`, and `test_data` have been normalized in memory, we will save these processed versions to new files within the `OUTPUT_DIR`. Subsequently, the global variables (`TRAIN_CSV`, `VAL_CSV`, `TEST_JSON`) will be updated to point to these new files, ensuring that all future operations in this notebook use the consistently formatted data.

In [ ]:
# Define new paths for the fully preprocessed files
normalized_train_path = os.path.join("/content/drive/MyDrive/output", "Arabic_train_preprocessed.jsonl")
normalized_val_path = os.path.join("/content/drive/MyDrive/output", "Arabic_val_preprocessed.jsonl")
normalized_test_path = os.path.join("/content/drive/MyDrive/output", "Arabic_test_preprocessed.json")

# Save to Drive
train_df.to_json(normalized_train_path, orient="records", lines=True)
val_df.to_json(normalized_val_path, orient="records", lines=True)

with open(normalized_test_path, "w") as f:
    json.dump(test_data, f, indent=4, ensure_ascii=False)

print(f"Fully preprocessed train data saved to: {normalized_train_path}")
print(f"Fully preprocessed val data saved to: {normalized_val_path}")
print(f"Fully preprocessed test data saved to: {normalized_test_path}")

Fully preprocessed train data saved to: /content/drive/MyDrive/output/Arabic_train_preprocessed.jsonl
Fully preprocessed val data saved to: /content/drive/MyDrive/output/Arabic_val_preprocessed.jsonl
Fully preprocessed test data saved to: /content/drive/MyDrive/output/Arabic_test_preprocessed.json


In [ ]:
import torch
import os

# Define the checkpoint to load
checkpoint_path = os.path.join(OUTPUT_DIR, 'model_epoch_0.pt')

if 'model' not in locals() or 'trainer' not in locals():
    print("ERROR: 'model' or 'trainer' is not defined. Please run cell 39fd019f first.")
elif os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check if the checkpoint file is a directory (meaning it's a saved model from save_pretrained)
    # or a .pt file (meaning it's a full checkpoint dict)
    if os.path.isdir(checkpoint_path): # This path will typically be a directory if saving a PEFT model
        # For PEFT models saved with save_pretrained, load the adapter weights
        # The base model should already be loaded in `model` instance
        # This part might need adjustment based on how the model was saved
        # For now, let's assume `model_epoch_0.pt` refers to a full state dict
        print(f"Warning: '{checkpoint_path}' is a directory. Attempting to load model_state_dict from a .pt file instead.")
        # Attempt to find the actual .pt file, assuming a standard naming convention
        actual_checkpoint_file = os.path.join(OUTPUT_DIR, 'checkpoint_epoch_0.pt') # Assuming `checkpoint_epoch_0.pt` exists
        if os.path.exists(actual_checkpoint_file):
            checkpoint = torch.load(actual_checkpoint_file, map_location=device)
        else:
            print(f"ERROR: Full checkpoint file '{actual_checkpoint_file}' not found.")
            checkpoint = None
    else: # It's a .pt file
        checkpoint = torch.load(checkpoint_path, map_location=device)

    if checkpoint and isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        trainer.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        trainer.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        trainer.history = checkpoint.get('history', trainer.history)
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming from epoch {start_epoch}")
    else:
        # Fallback: assume the file is just the state_dict
        print("Format mismatch or checkpoint not found: Loading file as raw state_dict.")
        # This line assumes checkpoint_path directly points to the model's state_dict
        # If it was a directory from save_pretrained, this would fail.
        # For now, assume it's a .pt file with just the state_dict.
        if checkpoint is not None: # Only attempt if checkpoint was successfully loaded as .pt
            model.load_state_dict(checkpoint)
        start_epoch = 1

    # Continue training from the determined start_epoch
    trainer.train(start_epoch=start_epoch)
else:
    print(f"No checkpoint found at: {checkpoint_path}")
    # Start training from scratch if no checkpoint is found
    trainer.train()

In [ ]:
import os
if os.path.exists(OUTPUT_DIR):
    print(f"Files in {OUTPUT_DIR}:")
    print(os.listdir(OUTPUT_DIR))
else:
    print(f"The directory {OUTPUT_DIR} does not exist. Please check if your Google Drive is mounted and the path is correct.")

The directory /content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models does not exist. Please check if your Google Drive is mounted and the path is correct.


In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get('login_key'))
train_df = pd.read_json(TRAIN_CSV, lines = True)

In [ ]:
print(train_df['Class'].unique())

[1 0]


In [16]:
with open(TEST_JSON, "r") as f:
  test_data = json.load(f)

predictions = []

evaluator = VerifierEvaluator(
    model_path=f"{OUTPUT_DIR}/checkpoint_epoch_0.pt",
    tokenizer_path=BASE_MODEL,
    base_model=BASE_MODEL,
    use_decomp=True,
)

# Example scoring
for idx, sample in enumerate(test_data):
  verdict_list = []
  verifier_score_list = []
  justification_list = []
  approved_majority_list = []
  for decoding_sample in range(1, len(sample["Reasoning_traces"]) + 1):
    justification = (
        remove_label_pattern(
            sample["Reasoning_traces"][decoding_sample - 1]
        ).split("Label:")[0]
    )
    score = evaluator.score(
    sample["claim"],
    sample.get("Questions", ""),
    sample["Verdict_list"][decoding_sample - 1].lower(),
    justification,
)
    verdict_list.append(sample["Verdict_list"][decoding_sample - 1])
    justification_list.append(justification)
    verifier_score_list.append(score)
    best_verdict = verdict_list[np.argmax(np.array(verifier_score_list))]
  predictions.append({
      "query_id": idx,
      "Claim": sample["claim"],
      "label": sample.get("label", best_verdict),
      "Verdict_BoN": best_verdict,
      "BoN_Verdict_list": verdict_list,
      "Reasoning_traces": justification_list,
      "score_list": verifier_score_list,
  })


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
len(test_data)

1600

In [17]:
import os

output_file_path = f"/content/drive/MyDrive/CheckThat_2026_TTS task/Training_reward_models/Data/Arabic/clef_predictions_final.json"
output_dir = os.path.dirname(output_file_path)

# Create the directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

with open(output_file_path, "w") as fp:
  json.dump(predictions, fp, indent=4)